In [ ]:
# --- Бібліотеки для даної роботи ---
try:
    import markdown, numpy, pandas, plotly, nbformat
    print("Бібліотеки вже встановлені. Пропускаємо інсталяцію.")
except ImportError:
    print("Встановлюємо бібліотеки...")
    %pip install markdown numpy pandas plotly nbformat

<div align="center">
    <h2><b>Architecture Design Document (ADD)</b></h2>
    <h1>Проектування глобальної E-Commerce платформи (Multi-Region)</h1>
    <h3>Глобальний маркетплейс «2BeMarket»</h3>
    <br>

**Метадані документа:**

| **Тип документа:** | High-Level Design (HLD) & Global Scale |
| :--- | :--- |
| **Статус:** | Draft / В розробці |
| **Масштаб:** | Global (Multi-Region Active-Active) |
| **Версія архітектури:** | 1.0.0 |
| **Дата створення:** | Березень 2026 р. |

<br>

**Автор проекту:**

| **Ім'я:** | Олег Гаценко |
| :--- | :--- |
| **Роль:** | Principal System Architect / Студент магістратури |
| **Організація:** | NeoVersity (Master's in AI & Machine Learning) |

</div>

---

### **Executive Summary (Резюме архітектури):**

Цей документ описує комплексне проектування високорівневої архітектури (HLD) для глобальної платформи електронної комерції **2BeMarket** (аналог Amazon / AliExpress). Платформа розрахована на понад **100 млн DAU** та роботу в умовах екстремальних навантажень (Global Black Friday). Головний архітектурний виклик — забезпечення мілісекундної затримки для покупців по всьому світу зі збереженням суворої консистентності фінансових транзакцій.

**Ключові архітектурні парадигми та рішення:**
* **Multi-Region Active-Active:** Розгортання інфраструктури у трьох незалежних географічних зонах (США, Європа, Азія) за допомогою Global Server Load Balancing (GSLB). Це гарантує виживання системи навіть у разі падіння цілого континенту (Disaster Recovery).
* **Conflict-Free Carts (CRDT):** Кошики користувачів реалізовані на базі безконфліктних структур даних (CRDT) у розподіленому NoSQL-сховищі (DynamoDB/Cassandra), що виключає втрату товарів при перемиканні користувача між регіонами.
* **Distributed Transactions & Saga:** Відмова від повільних 2PC-транзакцій на користь патерну **Saga** (Оркестрація) з використанням `Idempotency Keys`. Це гарантує надійність списання коштів та інвентаризації складських залишків.
* **Global Polyglot Persistence:** Використання NewSQL (наприклад, Google Spanner / CockroachDB) для транзакційного ядра з суворою консистентністю (ACID) на глобальному рівні, та ElasticSearch для миттєвого повнотекстового пошуку товарів.
* **Machine Learning & Personalization:** Асинхронний пайплайн (Kafka + Flink) для збору клікстріму та віддачі персоналізованих рекомендацій у режимі реального часу як для веб-клієнтів, так і для **мобільних застосунків**.
* **Multi-Cloud & Data Sovereignty:** Інфраструктура є Cloud-Agnostic (Kubernetes) і розгорнута в різних хмарах (AWS для Америки, GCP для Європи/Швейцарії, Alibaba для Азії). Це гарантує юридичну відповідність суворим місцевим законам про зберігання персональних даних (GDPR, FADP, PIPL).
* **Edge Security & Geo-Fencing:** Використання Cloudflare WAF на рівні глобального маршрутизатора для захисту від DDoS та жорсткого геоблокування (Geo-Ban) підсанкційних регіонів (наприклад, рф) ще до потрапляння трафіку у внутрішню мережу маркетплейсу.

### **Контекстна діаграма потоків (Level 0):**

```mermaid
graph LR
    %% Styling (Luxury & Dark Theme)
    classDef source fill:#2b2b2b,stroke:#00ffcc,stroke-width:2px,color:#fff
    classDef stream fill:#1a1a1a,stroke:#ff3366,stroke-width:2px,color:#fff
    classDef batch fill:#1a1a1a,stroke:#3399ff,stroke-width:2px,color:#fff
    classDef sink fill:#2b2b2b,stroke:#ffcc00,stroke-width:2px,color:#fff
    classDef banned fill:#3a0000,stroke:#ff0000,stroke-width:2px,color:#fff
    
    %% Нові Люксові Стилі
    classDef buyer_lux fill:#5e35b1,stroke:#ffd700,stroke-width:3px,color:#fff,rx:10px,ry:10px
    classDef seller_lux fill:#00796b,stroke:#ffd700,stroke-width:3px,color:#fff,rx:10px,ry:10px

    Buyer(("📱 Покупці<br>(Мобільний застосунок / Web)")):::buyer_lux
    Seller(("💻 Продавці<br>(Merchant Portal)")):::seller_lux
    Sanctioned(("🚫 Підсанкційний трафік<br>(рф тощо)")):::banned

    WAF{{"🛡️ Edge Security & WAF<br>(Cloudflare / Geo-Ban)"}}:::source
    GLB{{"🌐 Global Load Balancer<br>& API Gateway"}}:::source

    Catalog["🛒 Catalog & Search<br>(ElasticSearch + Redis)"]:::batch
    Checkout["💳 Cart & Checkout<br>(Saga Orchestrator)"]:::stream

    Bus[["⚡ Global Event Bus<br>(Apache Kafka)"]]:::source

    Storage[("🗄️ Polyglot Persistence<br>(NewSQL, Cassandra)")]:::sink
    Bank[["🏦 Платіжні Шлюзи<br>(Stripe/PayPal)"]]:::sink
    Logistics[["📦 Логістичні Хаби<br>(DHL/FedEx/Local Post)"]]:::sink

    %% Зв'язки безпеки та роутингу
    Buyer -- "Мільйони запитів/сек<br>(Пошук, Кошик)" --> WAF
    Seller -- "Управління інвентарем" --> WAF
    Sanctioned -. "Connection Dropped" .-> WAF
    WAF -- "Авторизований трафік" --> GLB

    %% Внутрішні потоки
    GLB -- "Read-heavy трафік" --> Catalog
    GLB -- "Write-heavy (Idempotent)" --> Checkout

    Checkout -- "Синхронна оплата" --> Bank
    Checkout -- "Асинхронні події" --> Bus

    Bus -- "Eventual Consistency" --> Storage
    Checkout -. "ACID Транзакції" .-> Storage
    Catalog -. "Індексація даних" .-> Storage
    Bus -- "Push: Інтеграція доставки" --> Logistics
```

<hr style="height: 15px; border: none; background: linear-gradient(to right, #007bff, #6610f2, #e83e8c, #fd7e14, #ffc107, #28a745); border-radius: 2px;">

## **1. Системні вимоги та Back-of-the-envelope аналіз**

Проект **2BeMarket** створюється як глобальна екосистема, що поєднує покупців та продавців у різних юрисдикціях. Тому вимоги до системи охоплюють не лише технічні, але й суворі юридичні та логістичні аспекти.

### 1.1. Функціональні вимоги (Functional Requirements)
1. **Каталог та Пошук:** Користувачі (**мобільний застосунок** / Web) можуть шукати товари, фільтрувати їх за десятками параметрів та переглядати деталі в режимі реального часу.
2. **Кошик та Оформлення (Checkout):** Можливість додавати товари в кошик, змінювати їх кількість та безшовно переходити до оплати через зовнішні платіжні шлюзи (Stripe, PayPal, Alipay).
3. **Управління замовленнями та Логістика:** Відстеження статусу замовлення, система повернень, а також інтеграція з логістичними хабами (Logistics Gateway) для оновлення трекінг-статусів.
4. **Управління користувачами та Продавцями:** Реєстрація покупців та B2B-портал для продавців (Merchant Portal) з управлінням інвентарем та зонами доставки.
5. **Персоналізація:** Генерація рекомендацій ("Часто купують разом") на основі історії покупок та клікстріму.
6. **Аналітика:** Надання агрегованої статистики продажів для продавців та інвесторів.

---

### 1.2. Нефункціональні вимоги (Non-Functional Requirements)
1. **Висока доступність (High Availability):** Цільова доступність 99.999% (не більше 5 хвилин простою на рік). Збереження працездатності навіть при падінні цілого AWS-регіону.
2. **Низька затримка (Low Latency):** Час відповіді API < 200 мс для 99% запитів; повнотекстовий пошук серед мільйонів товарів < 500 мс у будь-якій точці планети.
3. **Строга консистентність (Strong Consistency):** Гарантія ACID-транзакцій для фінансових операцій та списання залишків зі складу (уникнення проблеми подвійного продажу одного товару).
4. **Data Sovereignty & Комплаєнс:** Суворе дотримання законів про локалізацію персональних даних (GDPR в ЄС, FADP у Швейцарії, PIPL у Китаї). Дані користувачів не повинні перетинати кордони своїх регіонів без дозволу.
5. **Інтернаціоналізація (i18n):** Динамічна підтримка кількох мов, конвертація валют у реальному часі та автоматичний розрахунок регіональних податків (Tax Engine).

---

### 1.3. Оцінка навантаження та ресурсів (Capacity Planning)
Для доведення масштабованості системи наведемо базові (back-of-the-envelope) розрахунки для цільової аудиторії:

*   **Трафік:** 100 млн активних користувачів щодня (DAU). При середньому показнику 15 запитів на користувача маємо **1.5 млрд запитів/день**, що дорівнює базовому навантаженню у **~17,500 RPS** (Requests Per Second).
*   **Піковий RPS (Black Friday / Флеш-розпродажі):** Історично трафік у такі дні зростає в 10 разів. Система повинна бути спроектована на витримування **175,000 RPS**. З них приблизно 5% — це транзакції запису (Write), тобто **~8,750 TPS** (Transactions Per Second), що вимагає потужного брокера повідомлень (Kafka) та оптимізованого пулу підключень до БД.
*   **Пропускна здатність (Bandwidth):** Середній розмір відповіді API (JSON-метадані + посилання на CDN) складає близько 50 KB. Базовий вихідний трафік: 17,500 RPS × 50 KB ≈ **875 MB/sec** (близько 7 Gbps на рівні балансувальників).
*   **Зберігання даних (Storage):** Очікується близько 50 млн нових замовлень на день. Середній розмір об'єкта замовлення (JSON) ≈ 2 KB. 
    *   Приріст транзакційної БД: 50 млн × 2 KB = **100 GB/day**.
    *   За 5 років (з урахуванням 3-кратної реплікації для надійності): 100 GB × 365 × 5 × 3 = **~165 TB** високошвидкісного дискового простору лише для ядра транзакцій. Це математично обґрунтовує відмову від монолітного PostgreSQL на користь георозподілених NewSQL рішень (CockroachDB / Spanner).

**Візуалізація профілю навантаження під час пікових подій:**
```mermaid
pie title Розподіл трафіку (Black Friday - 175,000 RPS)
    "Перегляд каталогу та Пошук (Read-heavy)" : 85
    "Генерація рекомендацій (ML Inference)" : 10
    "Кошик та Оформлення (Write-heavy / ACID)" : 5
```